In [ ]:
# Script for styling the Jupyter Notebook using an external CSS file
from IPython.display import HTML

with open("style.css", "r") as f:
    css = f.read()

HTML(f"<style>{css}</style>")

# Random Forest

This notebook generates the report for random forest model which was trained using all available samples.

Results directory: `/home/mriveraceron/glv-research/tuning_results/RF_hpo`

Data directory: `/home/mriveraceron/glv-research/data_null`

# Numer of trees optimization

In [ ]:
import pandas as pd
from IPython.display import display, HTML

# Experiment directory
dir = '/home/mriveraceron/glv-research/tuning_results/RF_hpo'
df = pd.read_csv(f'{dir}/rf_results.csv')
display(df.sort_values(by="pearson_corr", ascending=False)
          .style.set_table_styles([{'selector': '', 'props': [('margin', '0 auto')]}]))

In [ ]:
# Elapsed time plot
import matplotlib.pyplot as plt

plt.clf()
plt.close('all')

# --- Global style ---
plt.rcParams.update({
    'font.family':      'DejaVu Sans',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.color':       '#e0e0e0',
    'grid.linewidth':   0.8,
    'grid.alpha':       0.7,
})

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#fafafa')

# ── Elapsed time plot ──────────────────────────────────────────
ax = axes[0]
ax.set_facecolor('#fafafa')

x = df['n_estimators']
y = df['elapsed(s)']

ax.plot(x, y, color='#2166ac', linewidth=2.2, zorder=3)
ax.fill_between(x, y, alpha=0.08, color='#2166ac')       # subtle area fill
ax.scatter(x, y, color='#2166ac', s=50, zorder=4)        # markers at each point

ax.set_xlabel('Número de árboles', labelpad=8, fontsize=10)
ax.set_ylabel('Tiempo transcurrido (segundos)', labelpad=8, fontsize=10)

# ── Metrics plot ───────────────────────────────────────────────
ax = axes[1]
ax.set_facecolor('#fafafa')

x     = df['n_estimators'].astype(str)
x_pos = range(len(x))

colors          = ['#E69F00', '#56B4E9', '#009E73']
metrics_to_plot = {'node_acc':      'VPP',
                   'pearson_corr': 'Corr. Pearson',
                   'spearman_corr':'Corr. Spearman'}

n       = len(metrics_to_plot)
width   = 0.25
offsets = [-width, 0, width]

for (m, lab), col, offset in zip(metrics_to_plot.items(), colors, offsets):
    bars = ax.bar(
        [p + offset for p in x_pos], df[m],
        width=width, label=lab, color=col,
        alpha=0.88, edgecolor='white', linewidth=0.6,
        zorder=3
    )
    ax.bar_label(bars, fmt='%.2f', fontsize=7.5, padding=3,rotation=90, color='#333333')

ax.set_xticks(x_pos)
ax.set_xticklabels(x, rotation=45, ha='right', fontsize=9)
ax.set_xlabel('Tamaño de muestra para entrenamiento', labelpad=8, fontsize=10)
ax.set_ylabel('Valor',                                labelpad=8, fontsize=10)
ax.set_ylim(0, ax.get_ylim()[1] * 1.18)               # headroom for bar labels
ax.legend(
    title='Rendimiento del modelo',
    loc='upper right', frameon=True,
    framealpha=0.5, edgecolor='#cccccc',
    fontsize=9, title_fontsize=10
)

# ── Title & layout ─────────────────────────────────────────────
for ax, letter in zip(axes, ['A', 'B']):
    ax.text(-0.05, 1.05, letter, transform=ax.transAxes, fontsize=14, fontweight='bold', va='bottom', ha='right')


plt.suptitle('Desempeño de Random Forest',
             fontsize=14, fontweight='bold', color='#222222', y=1.01)
plt.tight_layout()
save_path = '/home/mriveraceron/glv-research/tuning_results/RF_hpo/panel_perf.png'
print(f'Image saved at: {save_path}')
plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor="white")
plt.show()

## Caption
Figure 1. Random forest model performance. (A) Computational time elapsed as a function of the number of trees. (B) Performance metrics as a function of the number of trees.

## Interpretation
A Random Forest (RF) model was employed as a baseline method in which the adjacency matrix was deliberately excluded, allowing us to isolate and quantify the predictive contribution of the interaction network. As expected, the RF model failed to outperform the best-performing GNN model, confirming that the interaction network encodes meaningful information about the system.

Notably, increasing the number of estimators (trees) yielded only marginal improvements in performance, with no appreciable difference observed across configurations, suggesting the model reached its predictive ceiling regardless of ensemble size.

# Model predictions

The best-performing RF variant, achieving the highest Pearson correlation, used 900 estimator trees. Accordingly, only its predictions are presented below.

In [ ]:
import numpy as np
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import glob
from scipy.stats import pearsonr, spearmanr
import os

# Experiment directory
dir = '/home/mriveraceron/glv-research/tuning_results/RF_hpo'
data_path = f'{dir}/tree_params/rf_900_metrics.npz'

# Data for plotting
data = np.load(data_path)
mt = data['values_true']
mp  = data['values_pred']

# Compute correlations
r_p, _ = pearsonr(mt, mp)
r_s, _ = spearmanr(mt, mp)
x = np.log1p(mp)
y = np.log1p(mt)

# Hexbin
plt.close('all')
plt.clf()
fig, ax = plt.subplots(figsize=(5, 5))
hb = ax.hexbin(
    x, y,
    gridsize=45,
    cmap='YlOrRd',
    mincnt=1,
    bins='log',
    linewidths=0.15,
    edgecolors='none',      # cleaner look at high density
)
# Colorbar
cb = fig.colorbar(hb, ax=ax, pad=0.03, shrink=0.82, aspect=22)
cb.set_label('Recuento (escala log)', fontsize=11, labelpad=8)
cb.ax.tick_params(labelsize=10, direction='in')
cb.outline.set_linewidth(0.8)
# Identity line (y = x)
lims = [-0.02, 1.05]
ax.plot(lims, lims, color='0.3', linewidth=1.2, linestyle='--', alpha=0.85, zorder=5, label='Correlación perfecta')
# Labels, title
ax.set_xlabel('Keystoneness predicho', labelpad=8)
ax.set_ylabel('Keystoneness esperado', labelpad=8)
fig.suptitle('GraphConv — valores de keystoneness', fontsize=15, fontfamily='serif')
ax.set_title('Prueba de concepto', fontsize=11, color='0.4', pad=6)
# Figure limits
ax.set_xlim(lims)
ax.set_ylim(lims)
# Ticks
ax.xaxis.set_major_locator(ticker.MultipleLocator(0.2))
ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.05))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.05))
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

# Correlation annotation box
stats_text = (
    f'$r$ = {r_p:.3f}$\;$(Pearson)\n'
    f'$\\rho$ = {r_s:.3f}$\;$(Spearman)'
)
ax.text(
    0.05, 0.95, stats_text,
    transform=ax.transAxes,
    ha='left', va='top',
    fontsize=10,
    fontfamily='serif',
    bbox=dict(
        boxstyle='round,pad=0.5,rounding_size=0.4',
        facecolor='white',
        edgecolor='0.75',
        linewidth=0.8,
        alpha=0.9,
    ),
    zorder=10,
)
# Legend for identity line
ax.legend(loc='lower right', fontsize=10, framealpha=0.9, edgecolor='0.75')
# Clean spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f'{dir}/RF_best_preds.png')
plt.show()

# Confussion matrix

In [ ]:
# For confussion matrix we will require to compute the number of nodes
import torch
from pathlib import Path

# Count the number of nodes
eval_dir = Path('/home/mriveraceron/glv-research/data_null/efa83c9fafa0_eval')
data_tensor = [item for f in sorted(eval_dir.glob('*.pt')) for item in torch.load(f, weights_only= False)]
graph_nodes = np.array([s.x.shape[0] for s in data_tensor])
graph_nodes.sum()

# Generate the confussion matrix
idxt = data['max_idx_true']
idxp = data['max_idx_pred']

# Compute mask once, reuse for tp and fp
correct = idxt == idxp
tp = correct.sum()
fp = len(idxt) - tp          # avoids a second full pass over the array
fn = fp
tn = graph_nodes.sum() - len(graph_nodes) - fp   # vectorized sum(nodes - 1)
accuracy = (tp + tn) / (tp + tn + fp + fn)
ppv      = tp / (tp + fp)
print(f'RF model with 900 estimators | Accuracy: {accuracy:.4f} | PPV: {ppv:.4f} | By chance: {1/graph_nodes.mean():.4f}')
df_cm = pd.DataFrame(
    [[tp, fp], [fn, tn]],
    index   = ['Expected_Positive', 'Expected_Negative'],
    columns = ['Predicted_Positive', 'Predicted_Negative']
)
display(df_cm)
print('\n')
